# scGPT: Cell Type Annotation Fine-tuning Tutorial

이 노트북은 scGPT 공식 튜토리얼(`Tutorial_Annotation.ipynb`)을 기반으로,
**Google Colab T4 GPU**에서 실행 가능하도록 정리한 버전입니다.

**사전 요구사항**: Colab 런타임 → GPU (T4) 로 설정

- Paper: [Cui et al., Nature Methods 2024](https://doi.org/10.1038/s41592-024-02201-0)
- GitHub: [bowang-lab/scGPT](https://github.com/bowang-lab/scGPT)
- Dataset: MS (Multiple Sclerosis) brain single-cell RNA-seq

## 1. 환경 설정 (uv 가상환경)

Colab 기본 패키지 버전 충돌을 피하기 위해 `uv`로 `.venv` 가상환경을 생성하고, 호환되는 버전을 설치합니다.

In [ ]:
%%bash
# 1) uv 설치
pip install -q uv

# 2) .venv 가상환경 생성
uv venv .venv

# 3) PyTorch + torchtext (같은 인덱스에서 설치해야 .so 호환)
uv pip install --python .venv/bin/python torch torchvision torchaudio torchtext --index-url https://download.pytorch.org/whl/cu124

# 4) 과학 스택 + scGPT (numpy<2 로 numba 호환성 확보)
uv pip install --python .venv/bin/python "numpy<2" "scipy>=1.11,<1.14" numba scanpy anndata gdown scikit-learn matplotlib pandas

# 5) scGPT (--no-deps로 설치하여 torchtext 버전 덮어쓰기 방지)
uv pip install --python .venv/bin/python scgpt --no-deps

In [ ]:
import sys, os, glob

# .venv site-packages를 sys.path 최상위에 추가하여 Colab 기본 패키지 대신 사용
venv_site = glob.glob(".venv/lib/python*/site-packages")[0]
sys.path.insert(0, venv_site)

# 환경변수 업데이트
os.environ["VIRTUAL_ENV"] = os.path.abspath(".venv")
os.environ["PATH"] = os.path.abspath(".venv/bin") + ":" + os.environ["PATH"]

# 버전 확인
import numpy as np
print(f"numpy: {np.__version__}  (from {np.__file__})")
import scipy
print(f"scipy: {scipy.__version__}")
import torch
print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

import json
import copy
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import torch
from scipy.sparse import issparse
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch import nn
from torch.utils.data import Dataset, DataLoader

from scgpt.model import TransformerModel
from scgpt.tokenizer import tokenize_and_pad_batch, random_mask_value
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.preprocess import Preprocessor
from scgpt.utils import set_seed

set_seed(42)

# GPU 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. 데이터 및 모델 다운로드

MS(Multiple Sclerosis) 데이터셋과 scGPT 사전학습 체크포인트를 Google Drive에서 다운로드합니다.

In [ ]:
import gdown

# MS 데이터셋 다운로드
data_dir = "./ms_data"
os.makedirs(data_dir, exist_ok=True)
gdown.download_folder(
    "https://drive.google.com/drive/folders/1Qd42YNabzyr2pWt9xoY4cVMTAxsNBt4v",
    output=data_dir, quiet=False,
)

# scGPT_human 사전학습 체크포인트 다운로드
model_dir = "./scGPT_human"
os.makedirs(model_dir, exist_ok=True)
gdown.download_folder(
    "https://drive.google.com/drive/folders/1oWh_-ZRdhtoGQ2Fw24HP41FgLoomVo-y",
    output=model_dir, quiet=False,
)

print("\n다운로드 완료!")
print(f"데이터: {os.listdir(data_dir)}")
print(f"모델: {os.listdir(model_dir)}")

## 3. 데이터 로드 및 탐색

In [ ]:
# Reference 데이터 (학습용) + Query 데이터 (테스트용) 로드
adata_ref = sc.read_h5ad(f"{data_dir}/c_data.h5ad")
adata_query = sc.read_h5ad(f"{data_dir}/filtered_ms_adata.h5ad")

print(f"Reference: {adata_ref.shape[0]} cells x {adata_ref.shape[1]} genes")
print(f"Query:     {adata_query.shape[0]} cells x {adata_query.shape[1]} genes")

# Cell type 컬럼 통일
celltype_col = "Factor Value[inferred cell type - authors labels]"
adata_ref.obs["celltype"] = adata_ref.obs[celltype_col].astype(str)
adata_query.obs["celltype"] = adata_query.obs[celltype_col].astype(str)

# Batch 구분
adata_ref.obs["batch_id"] = "reference"
adata_query.obs["batch_id"] = "query"

# 유전자 이름 인덱스 설정
adata_ref.var.index = adata_ref.var["gene_name"].values
adata_query.var.index = adata_query.var["gene_name"].values

print(f"\nCell types (Reference):")
print(adata_ref.obs["celltype"].value_counts())

In [ ]:
# Reference + Query 결합 후 공통 유전자만 유지
common_genes = adata_ref.var.index.intersection(adata_query.var.index)
adata_ref = adata_ref[:, common_genes].copy()
adata_query = adata_query[:, common_genes].copy()

adata = adata_ref.concatenate(adata_query, batch_key="dataset")
print(f"결합 후: {adata.shape[0]} cells x {adata.shape[1]} genes")

## 4. Vocabulary 로드 및 유전자 필터링

사전학습 모델의 vocabulary에 포함된 유전자만 사용합니다.

In [ ]:
# Vocabulary 로드
vocab = GeneVocab.from_file(f"{model_dir}/vocab.json")
print(f"Vocabulary size: {len(vocab)}")

# Vocabulary에 special tokens 추가
special_tokens = ["<pad>", "<cls>", "<eoc>"]
for s in special_tokens:
    if s not in vocab:
        vocab.append_token(s)

# 데이터에서 vocab에 있는 유전자만 필터링
genes_in_vocab = [g for g in adata.var.index if g in vocab]
adata = adata[:, genes_in_vocab].copy()
print(f"Vocab 필터링 후: {adata.shape[0]} cells x {adata.shape[1]} genes")

## 5. 전처리 (Preprocessing)

scGPT 고유의 전처리: 정규화 → log1p → **51-bin 이산화(binning)**

In [ ]:
# HVG 선택 (seurat_v3는 raw count 필요)
sc.pp.highly_variable_genes(
    adata, n_top_genes=3000, flavor="seurat_v3", subset=True
)
print(f"HVG 선택 후: {adata.shape[0]} cells x {adata.shape[1]} genes")

# scGPT Preprocessor: normalize → log1p → binning
preprocessor = Preprocessor(
    use_key="X",
    filter_gene_by_counts=False,
    filter_cell_by_counts=False,
    normalize_total=1e4,
    result_normed_key="X_normed",
    log1p=True,
    result_log1p_key="X_log1p",
    subset_hvg=False,
    hvg_flavor="seurat_v3",
    binning=51,
    result_binned_key="X_binned",
)

preprocessor(adata)
print("전처리 완료 — binned 값 확인:")
print(f"  X_binned shape: {adata.layers['X_binned'].shape}")
print(f"  값 범위: {adata.layers['X_binned'].min()} ~ {adata.layers['X_binned'].max()}")

## 6. 학습/검증/테스트 분할 및 토큰화

In [ ]:
# Cell type → 정수 인코딩
cell_types = adata.obs["celltype"].unique().tolist()
cell_type_to_id = {ct: i for i, ct in enumerate(cell_types)}
id_to_cell_type = {i: ct for ct, i in cell_type_to_id.items()}
num_types = len(cell_types)
print(f"Cell types: {num_types}개")

adata.obs["celltype_id"] = adata.obs["celltype"].map(cell_type_to_id)

# Reference(학습+검증) / Query(테스트) 분할
train_adata = adata[adata.obs["batch_id"] == "reference"].copy()
test_adata = adata[adata.obs["batch_id"] == "query"].copy()

# 학습/검증 분할 (90/10)
train_idx, val_idx = train_test_split(
    range(train_adata.n_obs),
    test_size=0.1,
    random_state=42,
    stratify=train_adata.obs["celltype"],
)

print(f"Train: {len(train_idx)}, Valid: {len(val_idx)}, Test: {test_adata.n_obs}")

In [ ]:
# 토큰화 설정
max_seq_len = adata.shape[1] + 1  # +1 for <cls> token
pad_token = "<pad>"
pad_value = -2  # continuous input_emb_style에서의 pad value

# 유전자 토큰 ID 생성
gene_ids = np.array(vocab(adata.var.index.tolist()), dtype=int)

def prepare_tokenized_data(adata_subset):
    """AnnData에서 토큰화된 배치 데이터 생성"""
    binned_data = adata_subset.layers["X_binned"]
    if issparse(binned_data):
        binned_data = binned_data.toarray()

    tokenized = tokenize_and_pad_batch(
        binned_data,
        gene_ids,
        max_len=max_seq_len,
        vocab=vocab,
        pad_token=pad_token,
        pad_value=pad_value,
        append_cls=True,
        include_zero_gene=True,
    )
    return tokenized

# 학습/검증 데이터 토큰화
train_tokenized = prepare_tokenized_data(train_adata[train_idx])
val_tokenized = prepare_tokenized_data(train_adata[val_idx])

print(f"토큰화 완료")
print(f"  Gene IDs shape: {train_tokenized['genes'].shape}")
print(f"  Values shape:   {train_tokenized['values'].shape}")

## 7. Dataset & DataLoader 정의

In [ ]:
class CellDataset(Dataset):
    def __init__(self, tokenized_data, labels):
        self.gene_ids = tokenized_data["genes"]
        self.values = tokenized_data["values"]
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "gene_ids": self.gene_ids[idx],
            "values": self.values[idx],
            "label": self.labels[idx],
        }

# 라벨 준비
train_labels = train_adata[train_idx].obs["celltype_id"].values.astype(int)
val_labels = train_adata[val_idx].obs["celltype_id"].values.astype(int)

# Dataset & DataLoader
batch_size = 32
train_dataset = CellDataset(train_tokenized, train_labels)
val_dataset = CellDataset(val_tokenized, val_labels)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 8. 모델 초기화 및 사전학습 가중치 로드

In [ ]:
# 모델 설정 로드 (args.json)
with open(f"{model_dir}/args.json", "r") as f:
    model_configs = json.load(f)

print("모델 설정 (args.json):")
for k in ["embsize", "nheads", "d_hid", "nlayers", "n_layers_cls"]:
    print(f"  {k}: {model_configs.get(k, 'N/A')}")

embsize = model_configs["embsize"]
nhead = model_configs["nheads"]
d_hid = model_configs["d_hid"]
nlayers = model_configs["nlayers"]
nlayers_cls = model_configs.get("n_layers_cls", 3)

# TransformerModel 초기화
model = TransformerModel(
    ntoken=len(vocab),
    d_model=embsize,
    nhead=nhead,
    d_hid=d_hid,
    nlayers=nlayers,
    nlayers_cls=nlayers_cls,
    n_cls=num_types,
    vocab=vocab,
    dropout=0.2,
    pad_token=pad_token,
    pad_value=pad_value,
    n_input_bins=51,
    cell_emb_style="cls",
    input_emb_style="continuous",
    use_fast_transformer=False,  # Colab T4 호환성
)

# 사전학습 가중치 로드
model_file = f"{model_dir}/best_model.pt"
pretrained_dict = torch.load(model_file, map_location=device)
model_dict = model.state_dict()

# 크기가 맞는 가중치만 로드 (classification head는 새로 학습)
pretrained_dict = {
    k: v for k, v in pretrained_dict.items()
    if k in model_dict and v.shape == model_dict[k].shape
}
print(f"\n사전학습 가중치 로드: {len(pretrained_dict)}/{len(model_dict)} layers")
model_dict.update(pretrained_dict)
model.load_state_dict(model_dict)

model.to(device)
print(f"모델 파라미터: {sum(p.numel() for p in model.parameters()):,}")

## 9. Fine-tuning 학습 루프

In [ ]:
# 학습 설정
lr = 1e-4
epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.9)
scaler = torch.cuda.amp.GradScaler()  # Mixed Precision

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in loader:
        input_gene_ids = batch["gene_ids"].to(device)
        input_values = batch["values"].to(device)
        labels = batch["label"].to(device)

        src_key_padding_mask = input_gene_ids.eq(vocab[pad_token])

        with torch.cuda.amp.autocast():
            output_dict = model(
                input_gene_ids,
                input_values,
                src_key_padding_mask=src_key_padding_mask,
                CLS=True,
                CCE=False,
                MVC=False,
                ECS=False,
            )
            cls_output = output_dict["cls_output"]
            loss = criterion(cls_output, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * len(labels)
        preds = cls_output.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += len(labels)

    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in loader:
        input_gene_ids = batch["gene_ids"].to(device)
        input_values = batch["values"].to(device)
        labels = batch["label"].to(device)

        src_key_padding_mask = input_gene_ids.eq(vocab[pad_token])

        with torch.cuda.amp.autocast():
            output_dict = model(
                input_gene_ids,
                input_values,
                src_key_padding_mask=src_key_padding_mask,
                CLS=True,
                CCE=False,
                MVC=False,
                ECS=False,
            )
            cls_output = output_dict["cls_output"]
            loss = criterion(cls_output, labels)

        total_loss += loss.item() * len(labels)
        all_preds.extend(cls_output.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1, all_preds, all_labels

print("학습/평가 함수 정의 완료")

In [ ]:
# Fine-tuning 실행
best_val_acc = 0
best_model_state = None

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader)
    scheduler.step()

    print(
        f"Epoch {epoch+1:2d}/{epochs} | "
        f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = copy.deepcopy(model.state_dict())
        print(f"  → Best model 저장 (val_acc={val_acc:.4f})")

# Best model 복원
model.load_state_dict(best_model_state)
print(f"\n학습 완료! Best validation accuracy: {best_val_acc:.4f}")

## 10. 테스트 데이터 평가

In [ ]:
# 테스트 데이터 토큰화
test_tokenized = prepare_tokenized_data(test_adata)
test_labels = test_adata.obs["celltype_id"].values.astype(int)
test_dataset = CellDataset(test_tokenized, test_labels)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 테스트 평가
test_loss, test_acc, test_f1, test_preds, test_true = evaluate(model, test_loader)

print(f"={'=' * 50}")
print(f"Test Results (MS Query Dataset)")
print(f"={'=' * 50}")
print(f"  Accuracy:  {test_acc:.4f}")
print(f"  Macro F1:  {test_f1:.4f}")
print(f"  Loss:      {test_loss:.4f}")
print(f"={'=' * 50}")

# Classification Report
test_pred_names = [id_to_cell_type[p] for p in test_preds]
test_true_names = [id_to_cell_type[t] for t in test_true]
print("\nClassification Report:")
print(classification_report(test_true_names, test_pred_names, zero_division=0))

## 11. 시각화: UMAP + Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# --- Cell Embeddings 추출 (UMAP용) ---
@torch.no_grad()
def get_cell_embeddings(model, loader):
    model.eval()
    embeddings = []
    for batch in loader:
        input_gene_ids = batch["gene_ids"].to(device)
        input_values = batch["values"].to(device)
        src_key_padding_mask = input_gene_ids.eq(vocab[pad_token])

        with torch.cuda.amp.autocast():
            output_dict = model(
                input_gene_ids,
                input_values,
                src_key_padding_mask=src_key_padding_mask,
                CLS=True, CCE=False, MVC=False, ECS=False,
            )
        embeddings.append(output_dict["cell_emb"].cpu().numpy())
    return np.concatenate(embeddings, axis=0)

test_embeddings = get_cell_embeddings(model, test_loader)
print(f"Cell embeddings shape: {test_embeddings.shape}")

# AnnData에 embedding 저장 후 UMAP
test_adata.obsm["X_scgpt"] = test_embeddings
test_adata.obs["predicted"] = test_pred_names

sc.pp.neighbors(test_adata, use_rep="X_scgpt", n_neighbors=15)
sc.tl.umap(test_adata)

In [ ]:
# --- UMAP: Ground Truth vs Predicted ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sc.pl.umap(test_adata, color="celltype", title="Ground Truth", ax=axes[0], show=False,
           legend_fontsize=8, frameon=False)
sc.pl.umap(test_adata, color="predicted", title="scGPT Predicted", ax=axes[1], show=False,
           legend_fontsize=8, frameon=False)

plt.suptitle("scGPT Cell Type Annotation — MS Dataset", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("scgpt_umap.png", dpi=150, bbox_inches="tight")
plt.show()
print("UMAP 저장: scgpt_umap.png")

In [ ]:
# --- Confusion Matrix ---
# 빈도 높은 상위 N개 cell type만 표시 (가독성)
from collections import Counter

top_n = 10
ct_counts = Counter(test_true_names)
top_types = [ct for ct, _ in ct_counts.most_common(top_n)]

mask = [t in top_types for t in test_true_names]
filtered_true = [t for t, m in zip(test_true_names, mask) if m]
filtered_pred = [p for p, m in zip(test_pred_names, mask) if m]

cm = confusion_matrix(filtered_true, filtered_pred, labels=top_types)
fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(cm, display_labels=top_types)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, values_format="d")
ax.set_title(f"Confusion Matrix (Top {top_n} Cell Types)", fontsize=14)
plt.tight_layout()
plt.savefig("scgpt_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Confusion matrix 저장: scgpt_confusion_matrix.png")

## 12. 정리

### 이 노트북에서 다룬 내용
1. scGPT 사전학습 모델 다운로드 및 로드
2. MS 데이터셋 전처리 (HVG 선택 → binning)
3. Cell type annotation fine-tuning (CLS classification)
4. 테스트 데이터 평가 (Accuracy, Macro F1)
5. 시각화 (UMAP, Confusion Matrix)

### 참고 자료
- **논문**: [Cui et al., Nature Methods 2024](https://doi.org/10.1038/s41592-024-02201-0)
- **GitHub**: [bowang-lab/scGPT](https://github.com/bowang-lab/scGPT)
- **공식 튜토리얼**: [Tutorial_Annotation.ipynb](https://github.com/bowang-lab/scGPT/blob/main/tutorials/Tutorial_Annotation.ipynb)
- **블로그**: [BI Playground — scGPT 포스트](https://a7420174.github.io/ai/scGPT-Cell-Annotation/)